In [1]:
import math
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from tqdm import tqdm
from pathlib import Path

# Notebook: Create Training Windows Grid

This notebook creates a regular grid of training windows over the datastrips that have matching S2 candidates. The windows are 2560m x 2560m squares aligned to the coordinate system, and only windows that are fully contained within a datastrip are kept.

Purpose: Generate spatial windows for training data extraction from VHR and S2 products.

In [ ]:
CAND_PARQUET = "parquet/s2_vhr_candidates_10deg.parquet"
DATASSTRIP_GPKG = "gpkg/vhr2024_joined_acq_catalog.gpkg"

CRS_3035 = "EPSG:3035"
WINDOW_M = 2560

OUT_GPKG = Path("gpkg/windows_2560m_by_datastrip_from_candidates.gpkg")

## Load Candidate Datastrips

Load the list of datastrips that have S2 matches.

In [3]:
cand = pd.read_parquet(CAND_PARQUET)
ds_keep = sorted(cand["datastrip"].dropna().unique().tolist())
print(f"Datastrips in candidates: {len(ds_keep):,}")

Datastrips in candidates: 1,911


## Load Datastrip Geometries

Load the geometries of the selected datastrips and prepare them for processing.

In [ ]:
# -------------------------
# Load datastrip geometries
# -------------------------
ds = gpd.read_file(DATASSTRIP_GPKG)

if ds.crs is None:
    raise RuntimeError(f"{DATASSTRIP_GPKG} has no CRS set.")
if ds.crs.to_string().upper() != CRS_3035:
    ds = ds.to_crs(CRS_3035)

if "datastrip" not in ds.columns:
    raise RuntimeError("Expected a 'datastrip' column in vhr2024_joined_acq_catalog.gpkg")

ds = ds[ds["datastrip"].isin(ds_keep)].copy()
ds = ds.drop_duplicates(subset=["datastrip"])[["datastrip", "geometry"]].copy()
ds = ds[~ds.geometry.is_empty & ds.geometry.notna()].copy()

print(f"Datastrip geometries loaded: {len(ds):,}")

Datastrip geometries loaded: 1,911


## Build Grid Bounds

Calculate the bounding box and grid parameters for the window generation.

In [5]:
# -------------------------
# Build aligned grid over bbox of selected datastrips
# -------------------------
minx, miny, maxx, maxy = ds.total_bounds
minx = math.floor(minx / WINDOW_M) * WINDOW_M
miny = math.floor(miny / WINDOW_M) * WINDOW_M
maxx = math.ceil(maxx / WINDOW_M) * WINDOW_M
maxy = math.ceil(maxy / WINDOW_M) * WINDOW_M

nx = int((maxx - minx) // WINDOW_M)
ny = int((maxy - miny) // WINDOW_M)
print(f"Grid bbox (3035): {minx, miny, maxx, maxy}")
print(f"Grid candidates: {nx*ny:,} (nx={nx}, ny={ny})")

Grid bbox (3035): (-2711040, -3046400, 10032640, 5422080)
Grid candidates: 16,467,224 (nx=4978, ny=3308)


In [6]:
# Build all window polygons (candidate set)
geoms = []
rows = []
for iy in tqdm(range(ny), desc="Building candidate grid"):
    y0 = miny + iy * WINDOW_M
    y1 = y0 + WINDOW_M
    for ix in range(nx):
        x0 = minx + ix * WINDOW_M
        x1 = x0 + WINDOW_M
        geoms.append(box(x0, y0, x1, y1))
        rows.append({
            "window_id": f"W_{ix:05d}_{iy:05d}",
            "ix": ix,
            "iy": iy,
            "minx": float(x0),
            "miny": float(y0),
            "maxx": float(x1),
            "maxy": float(y1),
        })

win = gpd.GeoDataFrame(rows, geometry=geoms, crs=CRS_3035)
print(f"Built candidate windows: {len(win):,}")

Building candidate grid: 100%|██████████| 3308/3308 [07:39<00:00,  7.20it/s]


Built candidate windows: 16,467,224


In [9]:
# -------------------------
# Assign windows to datastrips by full containment
# -------------------------
# We want: window within datastrip
# This returns rows: each window that is within a datastrip, plus datastrip label
assigned = gpd.sjoin(win, ds, predicate="within", how="inner").drop(columns=["index_right"])
print(f"Windows assigned to datastrips: {len(assigned):,}")

Windows assigned to datastrips: 74,941


In [ ]:
# Ensure each window is assigned to exactly one datastrip.
counts = assigned.groupby("window_id").size()
unique_ids = counts[counts == 1].index
assigned_unique = assigned[assigned["window_id"].isin(unique_ids)].copy()

print(f"Windows within at least one datastrip: {len(assigned):,}")

Windows within at least one datastrip: 74,941


In [10]:
assigned_unique.to_file(OUT_GPKG, driver="GPKG")
print(f"Saved: {OUT_GPKG}")

Saved: windows_2560m_by_datastrip_from_candidates.gpkg


## Next Steps

The next step in the pipeline is to associate each VHR window with the most appropriate Sentinel-2 acquisition.

In practice, a single VHR datastrip may spatially overlap with multiple Sentinel-2 products. This typically occurs when the datastrip footprint lies close to the boundary between two Sentinel-2 swaths or acquisition times. As a result, windows originating from the same VHR datastrip may correspond to different Sentinel-2 products.

To handle this consistently, the next stage computes Sentinel-2 viewing geometry (e.g. VZA) and data masks at the window level. Windows are retained only if:
- all pixels within the window are covered by a valid Sentinel-2 acquisition, and
- no no-data pixels are present in the Sentinel-2 data mask.

By enforcing this constraint, we ensure that all retained windows:
- are fully contained within a single Sentinel-2 product, and
- are spatially and radiometrically consistent between VHR and Sentinel-2 inputs.

This filtering step guarantees that subsequent super-resolution experiments operate exclusively on fully valid, one-to-one window pairs, avoiding edge effects and mixed-product artifacts.